# 05 — Applications: Missile Paths

Every tool built across this course so far was building toward something. This notebook puts them together:

```
two points
    │
    ├─ haversine_km    →  range in kilometers
    ├─ compute_bearing →  launch azimuth in degrees
    └─ intersection    →  countries whose airspace is crossed
```

Given a launch point and a target, we can now produce a complete geometric picture of the engagement.

## Setup

In [ ]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "paths.geojson") as f:
    module_paths = json.load(f)

with open(DATA_DIR / "countries.geojson") as f:
    countries_fc = json.load(f)


# ── Distance & bearing (from modules 06 and 07) ──────────────────────────────

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi  = math.radians(lat2 - lat1)
    dlam  = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def compute_bearing(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dlam = math.radians(lon2 - lon1)
    x = math.sin(dlam) * math.cos(phi2)
    y = math.cos(phi1)*math.sin(phi2) - math.sin(phi1)*math.cos(phi2)*math.cos(dlam)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


# ── Intersection helpers (from notebooks 01–04) ───────────────────────────────

def orientation(p, q, r):
    return (q[0]-p[0])*(r[1]-q[1]) - (q[1]-p[1])*(r[0]-q[0])

def on_segment(p, q, r):
    return (min(p[0],r[0]) <= q[0] <= max(p[0],r[0]) and
            min(p[1],r[1]) <= q[1] <= max(p[1],r[1]))

def segments_intersect(p1, p2, p3, p4):
    d1=orientation(p3,p4,p1); d2=orientation(p3,p4,p2)
    d3=orientation(p1,p2,p3); d4=orientation(p1,p2,p4)
    if (d1>0 and d2<0 or d1<0 and d2>0) and (d3>0 and d4<0 or d3<0 and d4>0): return True
    if d1==0 and on_segment(p3,p1,p4): return True
    if d2==0 and on_segment(p3,p2,p4): return True
    if d3==0 and on_segment(p1,p3,p2): return True
    if d4==0 and on_segment(p1,p4,p2): return True
    return False

def polygon_edges(geometry):
    rings = geometry["coordinates"] if geometry["type"]=="Polygon" \
            else [ring for poly in geometry["coordinates"] for ring in poly]
    return [(ring[i], ring[i+1]) for ring in rings for i in range(len(ring)-1)]

def segment_bbox(p1, p2):
    return (min(p1[0],p2[0]),min(p1[1],p2[1]),max(p1[0],p2[0]),max(p1[1],p2[1]))

def geometry_bbox(geometry):
    rings = geometry["coordinates"] if geometry["type"]=="Polygon" \
            else [ring for poly in geometry["coordinates"] for ring in poly]
    coords = [pt for ring in rings for pt in ring]
    return (min(c[0] for c in coords), min(c[1] for c in coords),
            max(c[0] for c in coords), max(c[1] for c in coords))

def bboxes_overlap(b1, b2):
    return not (b1[2]<b2[0] or b2[2]<b1[0] or b1[3]<b2[1] or b2[3]<b1[1])

def path_crosses_feature(path_feature, country_feature):
    coords = path_feature["geometry"]["coordinates"]
    geom   = country_feature["geometry"]
    if geom is None: return False
    cbox  = geometry_bbox(geom)
    edges = polygon_edges(geom)
    for i in range(len(coords)-1):
        p1, p2 = coords[i], coords[i+1]
        if not bboxes_overlap(segment_bbox(p1, p2), cbox): continue
        for e1, e2 in edges:
            if segments_intersect(p1, p2, e1, e2): return True
    return False

def find_crossed_countries(path_feature, countries_fc):
    return [c for c in countries_fc["features"] if path_crosses_feature(path_feature, c)]

print(f"Loaded {len(module_paths['features'])} paths, {len(countries_fc['features'])} countries.")

## From Geometry to Intelligence

A single `path_report` function combines all three analyses. Given one path feature, it returns:

- **Range** — great-circle distance from launch to target
- **Bearing** — initial azimuth from launch point (0° = North, clockwise)
- **Airspace** — sorted list of countries whose borders the path crosses

In [ ]:
def path_report(path_feature, countries_fc):
    """
    Returns a dict with range_km, bearing_deg, and crossed_countries
    for a 2-point (or multi-point) path feature.
    """
    coords = path_feature["geometry"]["coordinates"]
    lon1, lat1 = coords[0]
    lon2, lat2 = coords[-1]

    crossed = find_crossed_countries(path_feature, countries_fc)

    return {
        "name":             path_feature["properties"]["name"],
        "origin":           path_feature["properties"]["origin"],
        "destination":      path_feature["properties"]["destination"],
        "range_km":         round(haversine_km(lon1, lat1, lon2, lat2)),
        "bearing_deg":      round(compute_bearing(lon1, lat1, lon2, lat2), 1),
        "crossed_countries": sorted(c["properties"]["name"] for c in crossed),
    }


# Run for all four paths
reports = [path_report(p, countries_fc) for p in module_paths["features"]]

for r in reports:
    print(f"\n{'─'*55}")
    print(f"  {r['name']}:  {r['origin']} → {r['destination']}")
    print(f"  Range:   {r['range_km']:,} km")
    print(f"  Bearing: {r['bearing_deg']}°")
    print(f"  Airspace ({len(r['crossed_countries'])} countries):")
    for country in r['crossed_countries']:
        print(f"    {country}")

## Visualizing One Path: Full Picture

A map that shows the path, the highlighted airspace, and the clean background together gives the complete picture in one view.

In [ ]:
def path_map(path_feature, countries_fc, path_color="#e74c3c", zoom=3, center=(35, 25)):
    """
    Returns a Map showing the path and its crossed countries highlighted.
    """
    crossed  = find_crossed_countries(path_feature, countries_fc)
    hit_names = {c["properties"]["name"] for c in crossed}

    hit_fc  = {"type": "FeatureCollection",
               "features": [c for c in countries_fc["features"]
                             if c["properties"]["name"] in hit_names]}
    miss_fc = {"type": "FeatureCollection",
               "features": [c for c in countries_fc["features"]
                             if c["properties"]["name"] not in hit_names]}

    m = Map(center=center, zoom=zoom, basemap=basemaps.CartoDB.Positron)
    m.add(GeoJSON(data=miss_fc,
                  style={"color": "#95a5a6", "fillColor": "#95a5a6",
                         "fillOpacity": 0.1, "weight": 0.4}))
    m.add(GeoJSON(data=hit_fc,
                  style={"color": path_color, "fillColor": path_color,
                         "fillOpacity": 0.5, "weight": 1.5}))
    m.add(GeoJSON(data={"type": "FeatureCollection", "features": [path_feature]},
                  style={"color": path_color, "weight": 3, "opacity": 0.95}))
    return m


# Show the Bravo path: Pyongyang → Honolulu
bravo = next(f for f in module_paths["features"] if f["properties"]["name"] == "Bravo")
path_map(bravo, countries_fc, path_color="#2980b9", zoom=2, center=(40, -100))

## Summary Table

A tabular summary is often more useful for briefing than individual maps. Build it with a simple loop.

In [ ]:
print(f"{'Name':<10} {'Origin → Destination':<38} {'Range':>8}  {'Bearing':>8}  {'Countries':>10}")
print("─" * 80)
for r in reports:
    label = f"{r['origin']} → {r['destination']}"
    print(f"{r['name']:<10} {label:<38} {r['range_km']:>7,}km  {r['bearing_deg']:>7}°  {len(r['crossed_countries']):>10}")

## Exercise A

Add a new path to the module dataset: a launch from **Beijing** (`[116.407, 39.904]`) to **Berlin** (`[13.405, 52.520]`) named `"Echo"`.

1. Build the Feature dict and append it to `module_paths["features"]`.
2. Call `path_report` on it and print the full result.
3. Display it with `path_map` using color `"#8e44ad"`.

In [ ]:
# 1. Build the Echo Feature and append to module_paths
# 2. Generate and print the path_report
# 3. Display with path_map
# Your code here

## Exercise B

Using all paths now in `module_paths` (including Echo from Exercise A):

1. Find the path with the **greatest range**.
2. Find the path that crosses the **most countries**.
3. Find any **country that appears in more than one path's airspace** — list those countries and which paths they appear in.

In [ ]:
# Make sure reports includes Echo before running this

# 1. Path with greatest range
# 2. Path crossing the most countries
# 3. Countries appearing in multiple paths' airspace
# Your code here

---

## Check Your Understanding

**1.** Why is intersection detection important for route and path analysis — what decisions does it support that distance and bearing alone cannot?

**2.** This module uses straight-line `LineString` paths. How would the intersection results change if the path curved along the actual great-circle arc, and in what direction would the error likely go?

```python
# No code needed — answer in your own words
```

## Next

Module 09 is complete. The next module introduces **spatial buffers** — expanding a point or path by a distance to model blast radius, exclusion zones, and proximity queries.